In [ ]:
#| default_exp core

# API

> API for ipykernel-helper

In [ ]:
#| export
from fastcore.meta import delegates
from fastcore.utils import patch,dict2obj
from types import ModuleType, FunctionType, MethodType, BuiltinFunctionType
from inspect import signature, currentframe
from functools import cmp_to_key,partial
from collections.abc import Mapping
from textwrap import dedent
from toolslm.funccall import *

import typing,warnings,re

from IPython.core.interactiveshell import InteractiveShell
from IPython.core.completer import ProvisionalCompleterWarning
from jedi import Interpreter, Script as jscript

from IPython.core.display import DisplayObject
from IPython.display import display,Markdown,HTML
from IPython.core.oinspect import Inspector

In [ ]:
#| export
warnings.filterwarnings('ignore', category=ProvisionalCompleterWarning)

In [ ]:
from pprint import pprint

In [ ]:
#| export
def _safe_repr(obj, max_len=200):
    "Safely get the repr() of an object, truncating if it exceeds max_len."
    try:
        s = str(obj)
        return s[:max_len] + ("…" if len(s)>max_len else "")
    except Exception as e: return f"<repr error: {str(e)}>"

In [ ]:
s = "Some long string that will be truncated"
print(_safe_repr(s, max_len=20))

Some long string tha…


In [ ]:
o = dict(name="Example", data=[1,2,3,4,5] * 5, nested={"a": 1, "b": 2, "c": [3, 4, 5] * 10})
print(_safe_repr(o, max_len=40))

{'name': 'Example', 'data': [1, 2, 3, 4,…


In [ ]:
#| export
@patch
def user_items(self:InteractiveShell, max_len=200, xtra_skip=()):
    "Get user-defined vars & funcs from namespace."
    ns,nsh = self.user_ns,self.user_ns_hidden
    ignore = {'nbmeta', 'receive_nbmeta'}
    ignore.add(xtra_skip)
    rm_types = (
        type, FunctionType, ModuleType, MethodType, BuiltinFunctionType,
        getattr(typing, '_SpecialGenericAlias', ()),
        getattr(typing, '_GenericAlias', ()),
        getattr(typing, '_SpecialForm', ())
    )
    user_items = {k:v for k, v in ns.items()
                  if not k in ignore and k not in nsh}
    user_vars = {k:_safe_repr(v, max_len=max_len)
                 for k, v in user_items.items() if not k.startswith('_') and not isinstance(v, rm_types)}
    user_fns = {k:str(signature(v)) for k, v in user_items.items()
                if isinstance(v, FunctionType) and v.__module__ == '__main__' and not k.startswith('__')}
    return user_vars,user_fns

In [ ]:
ipy = get_ipython()
_vs,_fs = ipy.user_items()
pprint(_vs)
print('---')
pprint(_fs)

{'custom_types': "{<class 'pathlib.Path'>}",
 'ipy': '<ipykernel.zmqshell.ZMQInteractiveShell object>',
 'o': "{'name': 'Example', 'data': [1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 1, 2, 3, 4, "
      "5, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5], 'nested': {'a': 1, 'b': 2, 'c': [3, "
      '4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5,…',
 's': 'Some long string that will be truncated',
 'user_items': 'None'}
---
{'_safe_repr': '(obj, max_len=200)'}


In [ ]:
#| export
def _rank(c, s):
    "Rank a completion `c` for text `s` with namespace `ns`."
    parts = s.split('.')
    is_public = not c.text.startswith('_')
    if c.type=='param': r=1
    elif c.mod=='__main__': r=2 # local
    elif len(parts)>1 and parts[0]==c.mod: r=3 # module
    elif c.mod=='builtins': r=4
    else: r=5
    return r if is_public else r+0.1

In [ ]:
#| export
@patch
def ranked_complete(self:InteractiveShell, code, line_no=None, col_no=None):
    ns = self.user_ns
    lines = code.splitlines(True)
    if line_no: offset = sum(len(lines[i]) for i in range(line_no-1)) + col_no -1
    else: offset = len(code)
    cs = self.Completer.completions(code, offset)
    def _c(a):
        res = dict2obj({attr: getattr(a, attr) for attr in dir(a) if attr[0]!='_'})
        res['mod']= getattr(ns.get(a.text, None), '__module__', None)
        res['rank'] = _rank(res, s=code)
        return res
    # Remove dunder vars, unless the user seems to be looking for them explicitly
    return [_c(c) for c in cs if not c.text.startswith('__') or '__' in code]

In [ ]:
from random import random

In [ ]:
def range_ex(
    a:str # some param
):
    "some func docstring"
    ...
ipy.ranked_complete('rang')

[{'end': 4,
  'signature': '',
  'start': 0,
  'text': 'range',
  'type': 'class',
  'mod': None,
  'rank': 5},
 {'end': 4,
  'signature': '(a: str)',
  'start': 0,
  'text': 'range_ex',
  'type': 'function',
  'mod': '__main__',
  'rank': 2}]

In [ ]:
res = ipy.ranked_complete('a="foo"\na.', 2, 3)
res[:2]

[{'end': 10,
  'signature': '() -> str',
  'start': 10,
  'text': 'capitalize',
  'type': 'function',
  'mod': None,
  'rank': 5},
 {'end': 10,
  'signature': '() -> str',
  'start': 10,
  'text': 'casefold',
  'type': 'function',
  'mod': None,
  'rank': 5}]

In [ ]:
#| export
def _signatures(ns, s, line, col):
    ctx = Interpreter(s, [ns]).get_signatures(line, col)
    if not ctx: ctx = jscript(s).get_signatures(line, col)
    return ctx

@patch
def sig_help(self:InteractiveShell, code, line_no=None, col_no=None):
    ns = self.user_ns
    ctx = _signatures(ns, code, line=line_no, col=col_no)
    def _s(s): return {'label':s.description,'typ':s.type, 'mod':s.module_name, 'doc':s.docstring(),
                       'idx':s.index, 'params':[{'name':p.name, 'desc':p.description} for p in s.params]}
    return [_s(opt) for opt in ctx]

In [ ]:
s = 'range('
res = ipy.sig_help(s, 1, len(s))
res[0]

{'label': 'class range',
 'typ': 'class',
 'mod': 'builtins',
 'doc': 'range(stop: int)\nrange(start: int, stop: int, step: int=...)\n\nrange(stop) -> range object\nrange(start, stop[, step]) -> range object\n\nReturn an object that produces a sequence of integers from start (inclusive)\nto stop (exclusive) by step.  range(i, j) produces i, i+1, i+2, ..., j-1.\nstart defaults to 0, and stop is omitted!  range(4) produces 0, 1, 2, 3.\nThese are exactly the valid indices for a list of 4 elements.\nWhen step is given, it specifies the increment (or decrement).',
 'idx': 0,
 'params': [{'name': 'stop', 'desc': 'param stop: int'}]}

In [ ]:
#| export
@patch
def get_vars(self:InteractiveShell, vs:list):
    "Get variables from namespace."
    ns = self.user_ns
    return {v:ns[v] for v in vs if v in ns}

In [ ]:
ipy.get_vars(['a', 's', "dummy"])

{'s': 'range('}

In [ ]:
#| export
def _get_schema(ns: dict, t):
    "Check if tool `t` has errors."
    if t not in ns: return f"`{t}` not found. Did you run it?"
    try: return get_schema(ns[t])
    except Exception as e: return f"`{t}`: {e}."

@patch
def get_schemas(self:InteractiveShell, fs:list):
    "Get schemas from namespace."
    ns = self.user_ns
    return {f:_get_schema(ns,f) for f in fs}

In [ ]:
ipy.get_schemas(['range_ex'])

{'range_ex': {'name': 'range_ex',
  'description': 'some func docstring',
  'input_schema': {'type': 'object',
   'properties': {'a': {'type': 'string', 'description': 'some param'}},
   'required': ['a']}}}

Errors are passed back as strings:

In [ ]:
def add(a:int,b:int): return a + b
ipy.get_schemas(['add'])

{'add': '`add`: Docstring missing!.'}

In [ ]:
ipy.get_schemas(['div'])

{'div': '`div` not found. Did you run it?'}

In [ ]:
#| export
@patch
def xpush(self:InteractiveShell, interactive=False, **kw):
    "Like `push`, but with kwargs"
    self.push(kw, interactive=interactive)

In [ ]:
ipy.push(dict(a=2))
a

2

In [ ]:
# ipykernel_helper version uses `**kwargs`
ipy.xpush(a=3)
a

3

The main benefits of using `ipy.push(dict(a=2))` over directly executing code are:

1. **Bulk variable assignment** - You can set multiple variables at once with a single command
2. **Programmatic variable injection** - It provides a way to inject variables into the namespace from another context or function
3. **No execution history** - Variables are added without creating an entry in the execution history
4. **No side effects** - It's a "pure" namespace modification without executing any code that might have side effects

There are several interesting functions in the IPython interpreter object that are useful for notebook development and interactive computing:

1. **`reset`/`reset_selective`** - Clear variables from the namespace (either all or selectively)
2. **`run_cell`/`run_cell_async`** - Execute code in a cell programmatically
3. **`set_next_input`** - Programmatically set the content of the next cell
4. **`system`/`system_raw`/`system_piped`** - Execute shell commands with different output handling
5. **`run_line_magic`/`run_cell_magic`** - Execute IPython magics programmatically
6. **`set_custom_exc`** - Set custom exception handlers

### Displaying MIME data

In [ ]:
cts = '#### A heading\n\nThis is **bold**.'
md_bundle = { 'text/markdown': cts }
ipy.display_pub.publish(data=md_bundle)

#### A heading

This is **bold**.

In [ ]:
#| export
@patch
def publish(self:InteractiveShell, data='', subtype='plain', mimetype='text', meta=None, update=False, **kw):
    if isinstance(data, DisplayObject): data,_ = self.display_formatter.format(data)
    elif not isinstance(data, Mapping): data = {f'{mimetype}/{subtype}': data}
    self.display_pub.publish(data, metadata=meta, transient=kw, update=update)

In [ ]:
ipy.publish(cts, 'markdown', foo='bar')

#### A heading

This is **bold**.

In [ ]:
ipy.publish(HTML('<b>hi</b> there'))

In [ ]:
ipy.publish({'text/plain':'hi there'})

hi there

In [ ]:
#| export
def transient(data='', subtype='plain', mimetype='text', meta=None, update=False, **kw):
    display({f'{mimetype}/{subtype}': data}, raw=True, metadata=meta, transient=kw, update=update)

In [ ]:
transient('hi there', foo='bar')

hi there

In [ ]:
transient('*hi* **there**', subtype='markdown')

*hi* **there**

In [ ]:
#| export
def run_cmd(cmd, data='', meta=None, update=False, **kw):
    transient(data, meta=meta, update=update, cmd=cmd, **kw)

In [ ]:
#| export
def get_md(cts):
    from html2text import HTML2Text
    h2t = HTML2Text(bodywidth=5000)
    h2t.ignore_links = False
    h2t.mark_code = True
    h2t.ignore_images = False
    res = h2t.handle(cts)
    def _f(m): return f'```\n{dedent(m.group(1))}\n```'
    return re.sub(r'\[code]\s*\n(.*?)\n\[/code]', _f, res or '', flags=re.DOTALL).strip()

In [ ]:
from bs4 import BeautifulSoup

In [ ]:
#| export
def read_url(
    url:str, # URL to read
    as_md:bool=True, # Convert HTML to Markdown?
    extract_section:bool=True, # If url has an anchor, return only that section
    selector:str=None # Select section(s) using BeautifulSoup.select (overrides extract_section)
):
    "Read URL and return contents"
    import httpx
    from urllib.parse import urlparse
    from bs4 import BeautifulSoup
    
    res = httpx.get(url, follow_redirects=True).raise_for_status().text
    if selector:
        sections = BeautifulSoup(res).select(selector)
        if sections: res = '\n\n'.join(str(section) for section in sections)
        else: res = ''
    elif extract_section:
        parsed = urlparse(url)
        if parsed.fragment:
            section = BeautifulSoup(res).find(id=parsed.fragment)
            if section: res = str(section)
    if as_md: return get_md(res)
    return res

In [ ]:
print(read_url('https://www.example.org'))

# Example Domain

This domain is for use in illustrative examples in documents. You may use this domain in literature without prior coordination or asking for permission.

[More information...](https://www.iana.org/domains/example)


In [ ]:
print(read_url('https://www.example.org', as_md=False, selector='body'))

<body>
<div>
<h1>Example Domain</h1>
<p>This domain is for use in illustrative examples in documents. You may use this
    domain in literature without prior coordination or asking for permission.</p>
<p><a href="https://www.iana.org/domains/example">More information...</a></p>
</div>
</body>


In [ ]:
print(read_url('https://fastcore.fast.ai/#getting-started'))

## Getting started

To install fastcore run: `conda install fastcore -c fastai` (if you use Anaconda, which we recommend) or `pip install fastcore`. For an [editable install](https://stackoverflow.com/questions/35064426/when-would-the-e-editable-option-be-useful-with-pip-install), clone this repo and run: `pip install -e ".[dev]"`. fastcore is tested to work on Ubuntu, macOS and Windows (versions tested are those shown with the `-latest` suffix [here](https://docs.github.com/en/actions/reference/specifications-for-github-hosted-runners#supported-runners-and-hardware-resources)).

`fastcore` contains many features, including:

  * `fastcore.test`: Simple testing functions
  * `fastcore.foundation`: Mixins, delegation, composition, and more
  * `fastcore.xtras`: Utility functions to help with functional-style programming, parallel processing, and more

To get started, we recommend you read through [the fastcore tour](https://fastcore.fast.ai/tour.html).


## Extension

In [ ]:
#| export
@patch
def _get_info(self:Inspector, obj, oname='', formatter=None, info=None, detail_level=0, omit_sections=()):
    "Custom formatter for ?? output"
    orig = self._orig__get_info(obj, oname=oname, formatter=formatter, info=info,
                               detail_level=detail_level, omit_sections=omit_sections)
    if detail_level==0: return orig
    info_dict = self.info(obj, oname=oname, info=info, detail_level=detail_level)
    out = []
    if c:=info_dict.get('source'): out.append(f"\n```python\n{dedent(c)}\n```")
    if c:=info_dict.get('file'): out.append(f"**File:** `{c}`")
    return {'text/markdown': '\n\n'.join(out), 'text/html': '', 'text/plain': orig['text/plain']}

In [ ]:
patch?

Signature: patch(f=None, *, as_prop=False, cls_method=False, set_prop=False)
Docstring: Decorator: add `f` to the first parameter's class (based on f's type annotations)
File:      ~/aai-ws/fastcore/fastcore/basics.py
Type:      function

In [ ]:
patch??


```python
def patch(f=None, *, as_prop=False, cls_method=False,  set_prop=False):
    "Decorator: add `f` to the first parameter's class (based on f's type annotations)"
    if f is None: return partial(patch, as_prop=as_prop, cls_method=cls_method, set_prop=set_prop)
    ann,glb,loc = get_annotations_ex(f)
    cls = union2tuple(eval_type(ann.pop('cls') if cls_method else next(iter(ann.values())), glb, loc))
    return patch_to(cls, as_prop=as_prop, cls_method=cls_method, set_prop=set_prop)(f)
```

**File:** `~/aai-ws/fastcore/fastcore/basics.py`

In [ ]:
#| export
def load_ipython_extension(ip):
    from ipykernel_helper import transient,run_cmd
    ns = ip.user_ns
    ns['read_url'],ns['transient'],ns['run_cmd'] = read_url,transient,run_cmd

## export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()